# INPUT_FILTER Block - Design Calculations

48V input filter for LLC resonant converter (120W, 36-60V range)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Design parameters
V_nom = 48  # Nominal input voltage
V_min = 36  # Minimum input
V_max = 60  # Maximum input
P_out = 120  # Output power (W)
f_sw_min = 100e3  # Min switching frequency
f_sw_max = 500e3  # Max switching frequency

## Bulk Capacitance Sizing

In [ ]:
C_bulk = 200e-6  # Total bulk capacitance (2× 100µF)

# Energy storage
E_stored = 0.5 * C_bulk * V_nom**2
print(f"Energy stored @ {V_nom}V: {E_stored*1e3:.1f} mJ")

# Holdup time (if input drops from max to min)
t_holdup = C_bulk * (V_max**2 - V_min**2) / (2 * P_out)
print(f"Holdup time (60V→36V): {t_holdup*1e3:.2f} ms")

# Impedance at switching frequencies
f_range = np.logspace(4, 6, 100)  # 10kHz to 1MHz
Z_bulk = 1 / (2 * np.pi * f_range * C_bulk)

plt.figure(figsize=(10, 4))
plt.loglog(f_range/1e3, Z_bulk*1e3, label='Bulk 200µF')
plt.axvline(f_sw_min/1e3, color='r', linestyle='--', label='Min f_sw')
plt.axvline(f_sw_max/1e3, color='r', linestyle='--', label='Max f_sw')
plt.xlabel('Frequency (kHz)')
plt.ylabel('Impedance (mΩ)')
plt.title('Bulk Capacitor Impedance vs Frequency')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"\nZ @ 100kHz: {1/(2*np.pi*100e3*C_bulk)*1e3:.1f} mΩ")
print(f"Z @ 500kHz: {1/(2*np.pi*500e3*C_bulk)*1e3:.1f} mΩ")

## Voltage Divider Analysis

In [ ]:
R1 = 47e3  # High side resistor
R2 = 3.3e3  # Low side resistor
V_adc_max = 3.3  # RP2040 ADC max voltage

# Divider ratio
ratio = R2 / (R1 + R2)
print(f"Divider ratio: {ratio:.4f}")

# Voltage sense across input range
V_in_range = np.linspace(0, 65, 100)
V_sense = V_in_range * ratio

plt.figure(figsize=(10, 4))
plt.plot(V_in_range, V_sense, label='V_sense')
plt.axhline(V_adc_max, color='r', linestyle='--', label='ADC max (3.3V)')
plt.axvline(V_min, color='g', linestyle=':', label='V_min (36V)')
plt.axvline(V_nom, color='b', linestyle=':', label='V_nom (48V)')
plt.axvline(V_max, color='orange', linestyle=':', label='V_max (60V)')
plt.xlabel('Input Voltage (V)')
plt.ylabel('ADC Sense Voltage (V)')
plt.title('Voltage Sense vs Input')
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim(0, 4.5)
plt.show()

print(f"\nV_sense @ {V_min}V: {V_min * ratio:.2f}V")
print(f"V_sense @ {V_nom}V: {V_nom * ratio:.2f}V")
print(f"V_sense @ {V_max}V: {V_max * ratio:.2f}V (EXCEEDS ADC MAX)")

# Find saturation point
V_in_saturate = V_adc_max / ratio
print(f"\nADC saturates above: {V_in_saturate:.1f}V input")

# Divider current
I_div = V_nom / (R1 + R2)
print(f"\nDivider current @ {V_nom}V: {I_div*1e3:.2f} mA")
P_div = V_nom**2 / (R1 + R2)
print(f"Divider power: {P_div*1e3:.1f} mW")

## ADC Filter Cutoff

In [ ]:
C_adc = 100e-9  # ADC filter capacitor

# RC filter with R2
f_cutoff = 1 / (2 * np.pi * R2 * C_adc)
print(f"ADC filter cutoff: {f_cutoff:.0f} Hz")

# Frequency response
f_range = np.logspace(0, 6, 1000)  # 1Hz to 1MHz
H = 1 / np.sqrt(1 + (f_range / f_cutoff)**2)
H_dB = 20 * np.log10(H)

plt.figure(figsize=(10, 4))
plt.semilogx(f_range, H_dB)
plt.axvline(f_cutoff, color='r', linestyle='--', label=f'f_c = {f_cutoff:.0f} Hz')
plt.axhline(-3, color='g', linestyle=':', label='-3dB')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Attenuation (dB)')
plt.title('ADC RC Filter Frequency Response')
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim(-60, 5)
plt.show()

# Attenuation at switching frequencies
atten_100k = -20 * np.log10(1 / np.sqrt(1 + (100e3 / f_cutoff)**2))
atten_500k = -20 * np.log10(1 / np.sqrt(1 + (500e3 / f_cutoff)**2))
print(f"\nAttenuation @ 100kHz: {atten_100k:.1f} dB")
print(f"Attenuation @ 500kHz: {atten_500k:.1f} dB")

## TVS Protection Analysis

In [ ]:
# SMBJ58A parameters (from datasheet)
V_br = 58  # Breakdown voltage
V_clamp = 93.6  # Clamping voltage @ 6.5A
I_surge = 6.5  # Surge current @ V_clamp
P_pulse = 600  # Pulse power rating (W)

print(f"TVS SMBJ58A:")
print(f"  Breakdown: {V_br}V")
print(f"  Clamp: {V_clamp}V @ {I_surge}A")
print(f"  Pulse power: {P_pulse}W (10/1000µs)")

# Voltage margins
margin_nom = (V_br - V_nom) / V_nom * 100
margin_max = (V_br - V_max) / V_max * 100
print(f"\nMargins:")
print(f"  vs {V_nom}V nominal: {margin_nom:.1f}%")
print(f"  vs {V_max}V max: {margin_max:.1f}%" if margin_max > 0 else f"  vs {V_max}V max: WARNING - exceeds breakdown!")

## Summary

### Design Values (Verified)
- **Bulk capacitance:** 200µF (2× 100µF SMD electrolytic)
- **Ceramic bypass:** ~22µF effective (2× 10µF + 2× 1µF, derated)
- **TVS protection:** SMBJ58A (58V breakdown, 93.6V clamp)
- **Voltage sense:** 47kΩ / 3.3kΩ (ratio 0.0655, 48V → 3.14V)
- **ADC filter:** 100nF (482Hz cutoff)

### Warnings
1. **ADC saturates above 50V input** - Telemetry only, not safety-critical
2. **Input >60V may exceed TVS breakdown** - Operating range must be enforced
3. **Holdup time minimal** (1.92ms) - LLC doesn't require it, but no ride-through capability

### What Still Needs Verification
- Actual LLC input ripple current (depends on resonant tank Q)
- EMI/conducted emissions performance
- TVS transient clamping under real surge conditions
- Layout effects on HF ceramic effectiveness